[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jairomelo/aiOCR/blob/main/models/Tesseract-OCR/Tesseract-OCR.ipynb)

## Prerequisites

Runtime: Python 3, **CPU only** — no GPU required.

```bash
brew install tesseract tesseract-lang   # binary + all language packs
```

**Local (Linux/Debian):**
```bash
sudo apt-get install tesseract-ocr tesseract-ocr-spa
```

**Google Colab:** the install cell below handles the binary automatically.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print("Not running in Colab — skipping Drive mount.")

Not running in Colab — skipping Drive mount.


In [2]:
import subprocess, sys

# Colab: install the Tesseract binary + Spanish language pack
# Local (Mac): `brew install tesseract tesseract-lang` (see Prerequisites)
# Local (Linux): `sudo apt-get install tesseract-ocr tesseract-ocr-spa`
try:
    import google.colab  # noqa: F401
    subprocess.run(
        ['apt-get', 'install', '-q', '-y', 'tesseract-ocr', 'tesseract-ocr-spa'],
        check=True,
    )
except ImportError:
    pass

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pytesseract', 'Pillow'],
    check=True,
)

/Users/jmeloflorez-local/Research/aiOCR/.venv/bin/python: No module named pip


CalledProcessError: Command '['/Users/jmeloflorez-local/Research/aiOCR/.venv/bin/python', '-m', 'pip', 'install', '-q', 'pytesseract', 'Pillow']' returned non-zero exit status 1.

In [3]:
import pytesseract
from PIL import Image

In [4]:
from pathlib import Path

# Colab
# WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')

# Local: walk up from cwd until we find pyproject.toml (project root marker).
# This is robust regardless of where VS Code starts the Jupyter kernel
# (notebook directory, workspace root, or elsewhere).
def _find_project_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(
        f"Could not find project root — no '{marker}' found above {start}.\n"
        "Set WORKING_DIR manually or check your working directory."
    )

WORKING_DIR = _find_project_root(Path.cwd())
print(f"WORKING_DIR: {WORKING_DIR}")

# Sanity-check: print the Tesseract version so we know the binary is found
print(f"Tesseract version: {pytesseract.get_tesseract_version()}")
print(f"Available languages: {pytesseract.get_languages()}")

WORKING_DIR: /Users/jmeloflorez-local/Research/aiOCR
Tesseract version: 5.5.2
Available languages: ['afr', 'amh', 'ara', 'asm', 'aze', 'aze_cyrl', 'bel', 'ben', 'bod', 'bos', 'bre', 'bul', 'cat', 'ceb', 'ces', 'chi_sim', 'chi_sim_vert', 'chi_tra', 'chi_tra_vert', 'chr', 'cos', 'cym', 'dan', 'deu', 'div', 'dzo', 'ell', 'eng', 'enm', 'epo', 'equ', 'est', 'eus', 'fao', 'fas', 'fil', 'fin', 'fra', 'frk', 'frm', 'fry', 'gla', 'gle', 'glg', 'grc', 'guj', 'hat', 'heb', 'hin', 'hrv', 'hun', 'hye', 'iku', 'ind', 'isl', 'ita', 'ita_old', 'jav', 'jpn', 'jpn_vert', 'kan', 'kat', 'kat_old', 'kaz', 'khm', 'kir', 'kmr', 'kor', 'kor_vert', 'lao', 'lat', 'lav', 'lit', 'ltz', 'mal', 'mar', 'mkd', 'mlt', 'mon', 'mri', 'msa', 'mya', 'nep', 'nld', 'nor', 'oci', 'ori', 'osd', 'pan', 'pol', 'por', 'pus', 'que', 'ron', 'rus', 'san', 'sin', 'slk', 'slv', 'snd', 'snum', 'spa', 'spa_old', 'sqi', 'srp', 'srp_latn', 'sun', 'swa', 'swe', 'syr', 'tam', 'tat', 'tel', 'tgk', 'tha', 'tir', 'ton', 'tur', 'uig', 'ukr', '

In [5]:
IMAGE_FILES = [
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_1.png',
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_46.png',
    WORKING_DIR / 'images/CO_18180627/CO_18180627_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_4.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_1.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_2.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_1.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_3.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_4.png',
    WORKING_DIR / 'images/AR_SR8V4R3/AR_SR8V4R3_4.jpg',
    WORKING_DIR / 'images/fpineda_30_pza2/fpineda_30_pza2_page_2.png',
    WORKING_DIR / 'images/fpineda_184_pza6/fpineda_184_pza6_page_1.png',
    WORKING_DIR / 'images/fpineda_196_pza8/fpineda_196_pza8_page_12.png'
]

## Inference

In [6]:
import time

transcription_out = WORKING_DIR / 'transcriptions/Tesseract-OCR'
transcription_out.mkdir(parents=True, exist_ok=True)

# --oem 1 : LSTM neural network engine (Tesseract 4/5, most accurate)
# --psm 1 : automatic page segmentation with orientation & script detection
#           Use --psm 3 (default, no OSD) if you get 'OSD' errors.
TESS_CONFIG = r'--oem 1 --psm 1'
LANG = 'spa'  # Spanish; change to 'eng', 'lat', or 'spa+eng' as needed

for IMAGE_FILE in IMAGE_FILES:
    image_stem = IMAGE_FILE.stem
    out_path = transcription_out / f'{image_stem}.md'

    if out_path.exists():
        print(f'Skipping (already done): {image_stem}')
        continue

    if not IMAGE_FILE.exists():
        print(f'Skipping (image not found): {IMAGE_FILE}')
        continue

    image = Image.open(IMAGE_FILE).convert('RGB')

    t0 = time.time()
    transcription = pytesseract.image_to_string(image, lang=LANG, config=TESS_CONFIG)
    elapsed = time.time() - t0

    out_path.write_text(transcription.strip(), encoding='utf-8')
    print(f'Done in {elapsed:.1f}s — saved: transcriptions/Tesseract-OCR/{image_stem}.md')

Skipping (already done): CCundinamarca_page_1
Skipping (already done): CCundinamarca_page_46
Skipping (already done): CO_18180627_page_1
Skipping (already done): dmcz_18250101_page_1
Skipping (already done): dmcz_18250101_page_4
Skipping (already done): el-redactor-1_page_1
Skipping (already done): el-redactor-1_page_2
Skipping (already done): pineda1_page_1
Skipping (already done): pineda1_page_3
Skipping (already done): pineda1_page_4
Skipping (already done): AR_SR8V4R3_4
Done in 0.9s — saved: transcriptions/Tesseract-OCR/fpineda_30_pza2_page_2.md
Done in 2.2s — saved: transcriptions/Tesseract-OCR/fpineda_184_pza6_page_1.md
Done in 1.6s — saved: transcriptions/Tesseract-OCR/fpineda_196_pza8_page_12.md


### Saving the output